# W86 Frontier-Scale Closure on Colab Pro (NVIDIA A100 / L4 / V100)

Closes the post-W83 P0 substrate blockers (#25, #26, the #27 hidden-state-intercept-at-32k axis) by loading **Llama-3.1-8B-Instruct** in bf16 on the Colab Pro GPU and running the W80 instrumentation contract end-to-end.

## Setup (do this once)

1. **Runtime:** *Runtime → Change runtime type → T4 / V100 / A100 GPU* (A100 is best; V100 or L4 also work; T4 will OOM at 32 k tokens — drop `--horizon-tokens` to 8192 if you have to).
2. **Secret:** Click the **🔑 key icon** in the left sidebar → *+ Add new secret* → name = `hf_token`, value = `hf_xxxxxxxx` (your HuggingFace read token), toggle *Notebook access* **on**.
3. **Llama license:** make sure you've accepted the Meta Llama-3.1 license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct (gated repo).
4. *Runtime → Run all*. Total wall-clock: ~30 minutes on A100, ~60 minutes on V100/L4.

## Outputs

* `frontier_closure_report.json` + per-issue sidecars are saved to `/content/w86_<TIMESTAMP>/`.
* The final cell copies them to `/content/drive/MyDrive/coordpy_frontier_closure/w86_<TIMESTAMP>/` so you can share them back without re-running.

## Anti-cheat preserved

* The closure code lives in the `coordpy` package on `origin/main`; this notebook only invokes it.
* The HF token lives in Colab's Secrets store, never in the repo.
* Nothing in the notebook computes the report; everything is delegated to `scripts/run_frontier_closure_w86.py`.

In [ ]:
# --- 1. Environment probe + workdir ---
import os, sys, subprocess, json, datetime
print('python:', sys.version)
RUN_TS = datetime.datetime.now(
    tz=datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('RUN_TS:', RUN_TS)
OUT_DIR = f'/content/w86_{RUN_TS}'
os.makedirs(OUT_DIR, exist_ok=True)
print('OUT_DIR:', OUT_DIR)
subprocess.run(['nvidia-smi'], check=False)
import torch
print('torch:', torch.__version__,
      'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  '
          f'mem={p.total_memory / (1024**3):.2f} GiB  '
          f'sm={p.major}.{p.minor}')

In [ ]:
# --- 2. Install transformers / accelerate ---
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0',
    'accelerate>=0.34.0',
    'huggingface_hub>=0.24.0',
], check=True)
import transformers, accelerate, huggingface_hub
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('huggingface_hub:', huggingface_hub.__version__)

In [ ]:
# --- 3. HF login via Colab Secrets ---
# Expects a Colab Secret named `hf_token` with notebook access on.
from google.colab import userdata  # type: ignore
hf_token = userdata.get('hf_token').strip()
assert hf_token.startswith('hf_'), (
    'Colab Secret `hf_token` must start with hf_. '
    'Add it via the key icon in the left sidebar.')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print('HF login OK; token len =', len(hf_token))

In [ ]:
# --- 4. Clone CoordPy from origin/main and install in editable mode ---
subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/adotdong29/CoordPy.git',
    '/content/CoordPy',
], check=True)
os.chdir('/content/CoordPy')
print('HEAD =', subprocess.check_output(
    ['git', 'log', '-1', '--oneline'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
], check=True)
import coordpy
print('coordpy:', coordpy.__version__,
      'SDK:', getattr(coordpy, 'SDK_VERSION', '?'))

In [ ]:
# --- 5. Run the closure driver (all three issues) ---
rc = subprocess.run([
    sys.executable,
    'scripts/run_frontier_closure_w86.py',
    '--out-dir', OUT_DIR,
    '--model-name', 'meta-llama/Llama-3.1-8B-Instruct',
    '--device', 'cuda:0',
    '--precision-tier', 'tier_bf16',
    '--seed', '86001001',
    '--n-train-prompts', '40',
    '--n-eval-prompts', '10',
    '--horizon-tokens', '32768',
], check=False)
print('closure driver exit code:', rc.returncode)

In [ ]:
# --- 6. Print the report summary ---
report_path = os.path.join(OUT_DIR, 'frontier_closure_report.json')
with open(report_path, 'rb') as fh:
    report = json.load(fh)
print('report_cid:', report.get('report_cid'))
print('model_load_error:', report.get('model_load_error', 'none'))
for k in ('closure_25', 'closure_26', 'closure_27'):
    c = report.get(k, {})
    if not c:
        continue
    print(f'\n=== {k} ===')
    for kk in (
            'conformance', 'hidden_state_intercept_bench',
            'replay_from_kv', 'w83_load_bearing_claim_reproduced',
            'live_strictly_beats_synthetic',
            'intercept_moves_cid_at_min_32k',
            'wall_seconds', 'wall_seconds_conformance',
            'wall_seconds_replay', 'wall_seconds_hidden_state_intercept',
            'error',
    ):
        if kk in c:
            v = c[kk]
            if isinstance(v, dict):
                print(f'  {kk}: {json.dumps(v)[:400]}')
            else:
                print(f'  {kk}: {v}')

In [ ]:
# --- 7. Re-verify the audit chain offline ---
rc = subprocess.run([
    sys.executable,
    'scripts/verify_w86_audit_chain.py',
    '--report', report_path,
], check=False)
print('\nverifier exit code:', rc.returncode)

In [ ]:
# --- 8. Copy results to Google Drive AND offer a zip download ---
# Drive save (preferred — survives runtime disconnects)
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)
DEST = f'/content/drive/MyDrive/coordpy_frontier_closure/w86_{RUN_TS}'
os.makedirs(DEST, exist_ok=True)
subprocess.run(
    f'cp -r {OUT_DIR}/. {DEST}/', shell=True, check=False)
print('saved to Drive:', DEST)
subprocess.run(f'ls -la {DEST}', shell=True, check=False)

# Zip + immediate download (backup)
zip_path = f'/content/w86_{RUN_TS}.zip'
subprocess.run(
    ['zip', '-rq', zip_path, f'w86_{RUN_TS}'],
    cwd='/content', check=False)
print('\nZip ready at', zip_path,
      '(size:',
      f'{os.path.getsize(zip_path) / 1024:.1f} KiB)')
from google.colab import files  # type: ignore
files.download(zip_path)